# Logic-Zero · Task 13–15: GRPO training

Continues from the DPO checkpoint with **G**roup **R**elative **P**olicy **O**ptimization — the model generates K rollouts per prompt, scores each via a rule-based reward (format + correctness + length), and updates toward higher-reward completions.

**Two-stage flow (cheap → expensive):**

1. **Stage A — Smoke test** (~10 min, ~1 unit on L4). Verifies the trl LoRA-as-reference pattern doesn't OOM and one training step runs cleanly. If it fails, STOP.
2. **Stage B — Main training** (~6–8h on L4, ~30 units). Only run after Stage A passes. Reduced scope (1000 prompts, 2 epochs).

**Required:** DPO checkpoint at `/content/drive/MyDrive/logic-zero/checkpoints/dpo/best/`.


## 1. Verify GPU + clean state

In [ ]:
import torch, gc
gc.collect()
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU — Runtime → Change runtime type → GPU')
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
assert free / 1e9 > 10, 'Less than 10GB free — restart runtime first'

## 2. Pull latest repo

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

## 3. Install dependencies

If you hit `MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES` ImportError after this cell, run `os.kill(os.getpid(), 9)` to restart the runtime, then start from cell 1 again.

In [ ]:
import subprocess
def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(['pip', 'install', '-q', '--default-timeout=180'] + specs,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed: {specs}')
pip_inst(['transformers>=4.46,<4.50', 'trl>=0.13.0,<0.15.0',
          'peft>=0.13.0,<0.16.0', 'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv', 'wandb', 'pytest'])
import transformers, peft, trl, z3
print(f'transformers={transformers.__version__}  peft={peft.__version__}  trl={trl.__version__}')

## 4. Secrets (WANDB optional, recommended for reward curve)

In [ ]:
from google.colab import userdata
for key in ['WANDB_API_KEY', 'HF_TOKEN']:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'✓ {key} loaded')
    except Exception:
        print(f'⚠ {key} not in Colab Secrets (optional)')
if 'WANDB_API_KEY' not in os.environ:
    os.environ['WANDB_DISABLED'] = 'true'
    print('wandb disabled (no key — training still runs)')

## 5. Mount Drive + restore DPO checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/logic-zero'
!mkdir -p {DRIVE_ROOT}/checkpoints/dpo {DRIVE_ROOT}/checkpoints/grpo {DRIVE_ROOT}/data
assert os.path.isdir(DRIVE_ROOT), 'Drive mount failed'
print(f'✓ Drive root: {DRIVE_ROOT}')

In [ ]:
import shutil
dpo_local = 'results/checkpoints/dpo/best'
dpo_drive = f'{DRIVE_ROOT}/checkpoints/dpo/best'
if not os.path.isdir(dpo_local):
    assert os.path.isdir(dpo_drive), f'NO DPO checkpoint in Drive: {dpo_drive}'
    os.makedirs(os.path.dirname(dpo_local), exist_ok=True)
    shutil.copytree(dpo_drive, dpo_local)
    print(f'✓ Restored DPO checkpoint: {dpo_local}')
adapter_file = f'{dpo_local}/adapter_model.safetensors'
size_mb = os.path.getsize(adapter_file) / 1e6
assert size_mb > 10, f'DPO adapter too small ({size_mb}MB) — likely corrupted'
print(f'DPO adapter: {size_mb:.1f}MB  files: {sorted(os.listdir(dpo_local))}')

## 6. Generate GRPO data (~5 min)

GRPO doesn't need labels — just puzzles + ground truth (used by the reward function).
Generates ~1780 prompts; we'll subsample to 1000 at training time via `--limit`.

In [ ]:
!mkdir -p data logs
grpo_data_local = 'data/grpo_data.jsonl'
grpo_data_drive = f'{DRIVE_ROOT}/data/grpo_data.jsonl'
if not os.path.isfile(grpo_data_local):
    if os.path.isfile(grpo_data_drive):
        shutil.copy(grpo_data_drive, grpo_data_local)
        print(f'✓ Restored {grpo_data_drive} → {grpo_data_local}')
    else:
        print('Generating fresh grpo_data.jsonl ...')
        !python -m data.gen_grpo_data --out {grpo_data_local}
        shutil.copy(grpo_data_local, grpo_data_drive)
        print(f'✓ Mirrored to {grpo_data_drive}')
n_lines = sum(1 for _ in open(grpo_data_local))
print(f'GRPO data: {n_lines} prompts')

---

## ⚠️ STAGE A — Smoke test (DO THIS FIRST)

Verifies the trl LoRA-as-reference pattern works on this GPU with this trl version.
**Runs 1 training step on 4 dummy prompts. ~10 min, ~1 compute unit on L4.**

**If this fails:** STOP. Debug before Stage B (which burns 30+ units).

Common failures:
- **OOM** → reduce `num_generations` or `max_completion_length` in Stage B
- **TypeError on tokenizer/processing_class** → trl version mismatch
- **`MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES`** → run `import os; os.kill(os.getpid(), 9)` then re-run from cell 1

---

In [ ]:
import time
t0 = time.time()
print(f'Smoke test started at {time.strftime("%H:%M:%S")} ...', flush=True)
ret = subprocess.run(
    ['pytest', 'train/test_grpo_smoke.py', '-v', '-s'],
    capture_output=True, text=True
)
print(ret.stdout[-3000:])
print(ret.stderr[-1500:])
print(f'\n--- Smoke test finished in {time.time()-t0:.0f}s ---')
SMOKE_PASSED = (ret.returncode == 0)
print(f'\n{"✓ SMOKE TEST PASSED — safe to proceed" if SMOKE_PASSED else "✗ SMOKE TEST FAILED — DO NOT proceed"}')
assert SMOKE_PASSED, 'Smoke test failed. Inspect output above; do not run Stage B.'

### Mini-smoke (5 real GRPO steps, recommended)

The pytest above uses dummy "hi" prompts. This runs ~5 real GRPO steps on K&K puzzles to surface reward-signal issues (e.g. all-zero rewards) before the full run.

**Expected:** `reward` column > 0 in logging output. If reward is 0 across all 5 steps, the parser isn't matching the model's output — STOP and investigate.

In [ ]:
t0 = time.time()
print('Mini-smoke: ~5 GRPO steps on real puzzles ...', flush=True)
!python -u -m train.grpo \
    --start-adapter {dpo_local} \
    --ref-adapter {dpo_local} \
    --grpo-data {grpo_data_local} \
    --out-dir /tmp/grpo_mini \
    --limit 10 \
    --epochs 1 \
    --batch-size 1 --grad-accum 1 \
    --num-generations 2 \
    --max-completion-length 256 \
    --run-name logic-zero-grpo-mini \
    2>&1 | tee logs/grpo_mini.log
print(f'\n--- Mini-smoke finished in {time.time()-t0:.0f}s ---')

### Decision point

Look at the mini-smoke log above:

- ✅ **`reward` column trending > 0** + no OOM + no crash → **proceed to Stage B**
- ❌ **`reward` ≈ 0 every step** → reward parser issue; STOP and check `train/reward.py`
- ❌ **OOM** → in Stage B reduce `--num-generations 4 → 2` and `--max-completion-length 512 → 384`
- ❌ **Anything else weird** → STOP and report the log before continuing

If anything is off, **don't run Stage B** — you'll burn 30+ units on a broken run.

---

## STAGE B — Main GRPO training

**Run only after Stage A passes.**

Config (budget-aware):
- **1000 prompts** (via `--limit 1000`)
- **2 epochs**
- batch=2 × grad-accum=4 (effective 8)
- num-generations=4, max-completion-length=512
- LR=1e-6, β=0.04 (KL penalty)

**Expected duration on L4:** 6–8h. Mirrors to Drive on completion.

**Keep-alive (paste in browser DevTools console, F12):**
```javascript
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
```

In [ ]:
DRIVE_GRPO = f'{DRIVE_ROOT}/checkpoints/grpo'
import time
print(f'Stage B started at: {time.strftime("%Y-%m-%d %H:%M:%S")}', flush=True)
!python -u -m train.grpo \
    --start-adapter {dpo_local} \
    --ref-adapter {dpo_local} \
    --grpo-data {grpo_data_local} \
    --out-dir results/checkpoints/grpo \
    --limit 1000 \
    --epochs 2 \
    --batch-size 2 --grad-accum 4 \
    --num-generations 4 \
    --max-completion-length 512 \
    --lr 1e-6 --beta 0.04 \
    --run-name logic-zero-grpo-main \
    --drive-backup-dir {DRIVE_GRPO} \
    2>&1 | tee logs/grpo_train.log

## Verify GRPO checkpoint

In [ ]:
grpo_local = 'results/checkpoints/grpo/best'
grpo_drive_best = f'{DRIVE_GRPO}/best'
assert os.path.isdir(grpo_local), f'GRPO training failed — no checkpoint at {grpo_local}'
files = sorted(os.listdir(grpo_local))
size_mb = os.path.getsize(f'{grpo_local}/adapter_model.safetensors') / 1e6
print(f'Local GRPO checkpoint: {size_mb:.1f}MB, files={files}')
if os.path.isdir(grpo_drive_best):
    drive_size = os.path.getsize(f'{grpo_drive_best}/adapter_model.safetensors') / 1e6
    print(f'✓ Drive backup OK: {drive_size:.1f}MB at {grpo_drive_best}')
else:
    print(f'⚠ Drive backup missing — run: !cp -r {grpo_local} {grpo_drive_best}')

## Smoke test on a real puzzle

In [ ]:
import json
from peft import PeftModel
from train.common import load_base_model, to_chat, extract_answer
import gc, torch
gc.collect(); torch.cuda.empty_cache()

model, tok = load_base_model()
model = model.to('cuda')
model = PeftModel.from_pretrained(model, grpo_local)
model.eval()
model.config.use_cache = True

dev = [json.loads(l) for l in open('data/dev_data.jsonl')]
rec = dev[5]
print('PUZZLE:\n' + rec['puzzle'][:300])
print('\nGT:', rec['ground_truth'])

with torch.no_grad():
    inp = tok(to_chat(tok, rec['puzzle']), return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=600, do_sample=False,
                          pad_token_id=tok.eos_token_id,
                          stop_strings=['</answer>'], tokenizer=tok,
                          temperature=None, top_p=None, top_k=None)
resp = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
print('\nRESPONSE:\n' + resp)
print(f'\nextracted: {extract_answer(resp, n=len(rec["ground_truth"]))}')

## ✅ Task 13–15 done

**Outputs:**
- `results/checkpoints/grpo/best/` — GRPO LoRA adapter (local + Drive backup)
- `data/grpo_data.jsonl` — training prompts
- `logs/grpo_train.log` — full training log

### Next: GRPO eval

In `02_eval.ipynb`:
- cell 2: `MODEL_KIND = 'grpo'`
- Run on **T4 free** to save units (~16h)

### Decision criteria
- GRPO eval ≥ DPO eval + 3pp → success
- GRPO eval flat/worse → likely reward hacking; consider think-length floor in reward
- GRPO eval much worse → revert and skip GRPO in writeup